# 006 Write Graph To Neo4j

这是 RAG 知识库学习线的第六课。

上一课产物：

```text
triples.json
```

本课目标：

```text
triples.json
-> Document 节点
-> Chunk 节点
-> Entity 节点
-> RELATION 关系
-> Neo4j 查询验证
```

学习目标：

1. 理解 GraphRAG 为什么需要图数据库。
2. 设计 Document / Chunk / Entity / RELATION 图模型。
3. 使用 Neo4j Python Driver 连接图数据库。
4. 使用 Cypher `MERGE` 保证重复运行不会无限重复创建节点。
5. 把三元组和证据写入 Neo4j。
6. 查询实体的一跳关系，为下一课图检索做准备。

注意：本课默认不自动写入 Neo4j。确认数据后，把 `RUN_NEO4J_WRITE` 改为 `True`。

## 1. 本课的位置

当前阶段：

```text
chunks.json
-> triples.json
-> Neo4j graph
```

ES 解决“哪些文本片段相关”，Neo4j 解决“实体之间有什么关系，以及这些关系来自哪个证据 chunk”。

## 2. 图模型设计

第一版图模型保持简单：

```text
(:Document {doc_id, file_name})
(:Chunk {chunk_id, doc_id, page_start, page_end, text})
(:Entity {name})

(:Document)-[:HAS_CHUNK]->(:Chunk)
(:Chunk)-[:MENTIONS]->(:Entity)
(:Entity)-[:RELATION {type, evidence, confidence, evidence_chunk_id, doc_id}]->(:Entity)
```

为什么关系上要保存 `evidence_chunk_id`？因为图谱里的关系不能只是结论，还必须能回到原文证据。

## 3. 导入依赖

本课使用：

```text
neo4j -> Neo4j Python Driver
json  -> 读取 triples.json
```

In [12]:
import importlib.metadata
import json
import os
from hashlib import sha1
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from neo4j import GraphDatabase

print('neo4j', importlib.metadata.version('neo4j'))

neo4j 6.2.0


## 4. 加载 chunks 和 triples

正常链路应该读取第五课生成的：

```text
notebooks/rag/generated/{doc_id}/triples.json
```

如果你还没有运行第五课，本课会创建一个小型 demo triple，方便先学习 Neo4j 写入结构。

In [13]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / 'requirements.txt').exists() and (path / 'notebooks').exists():
            return path
    return current

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env', override=False)

SAMPLE_PDF = PROJECT_ROOT / 'raw' / '北京市密云水库防御洪水方案.pdf'
if not SAMPLE_PDF.exists():
    raise FileNotFoundError(SAMPLE_PDF)

doc_id = sha1(SAMPLE_PDF.read_bytes()).hexdigest()[:16]
generated_dir = PROJECT_ROOT / 'notebooks' / 'rag' / 'generated' / doc_id
chunks_json_path = generated_dir / 'chunks.json'
triples_json_path = generated_dir / 'triples.json'

if not chunks_json_path.exists():
    raise FileNotFoundError(f'请先执行第三课生成 chunks.json: {chunks_json_path}')

chunks = json.loads(chunks_json_path.read_text(encoding='utf-8'))
chunks_by_id = {chunk['chunk_id']: chunk for chunk in chunks}

if triples_json_path.exists():
    triple_payload = json.loads(triples_json_path.read_text(encoding='utf-8'))
    triples = triple_payload.get('triples', [])
    print('loaded triples.json')
else:
    first_chunk = chunks[0]
    triples = [
        {
            'subject': '北京市密云水库',
            'predicate': '涉及方案',
            'object': '防御洪水方案',
            'evidence': first_chunk['text'][:120].replace('\n', ' '),
            'confidence': 0.5,
            'doc_id': first_chunk['doc_id'],
            'chunk_id': first_chunk['chunk_id'],
            'page_start': first_chunk['page_start'],
            'page_end': first_chunk['page_end'],
            'file_name': first_chunk['file_name'],
        }
    ]
    print('triples.json not found, using demo triple')

print('doc_id:', doc_id)
print('chunk_count:', len(chunks))
print('triple_count:', len(triples))
pprint(triples[:2])

loaded triples.json
doc_id: 63b7d4d0675426b5
chunk_count: 143
triple_count: 23
[{'chunk_id': '63b7d4d0675426b5_chunk_0024',
  'confidence': 0.95,
  'doc_id': '63b7d4d0675426b5',
  'evidence': '结合3 日内气象预报结果，开展洪水预报作业，根据洪',
  'file_name': '北京市密云水库防御洪水方案.pdf',
  'object': '3 日内气象预报结果, 阐述3 日内气象预报结果',
  'page_end': 19,
  'page_start': 19,
  'predicate': '结合',
  'subject': '3 年一遇洪水调度示例'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0024',
  'confidence': 0.9,
  'doc_id': '63b7d4d0675426b5',
  'evidence': '3 年一遇洪水控泄流量200m3/s, 最高水位152.93m, 最高水位出现在207h',
  'file_name': '北京市密云水库防御洪水方案.pdf',
  'object': '200m3/s, 最高水位152.93m, 最高水位出现在207h',
  'page_end': 19,
  'page_start': 19,
  'predicate': '结合',
  'subject': '3 年一遇洪水控泄流量'}]


## 5. 连接 Neo4j

当前教学环境：

```text
Neo4j Browser: http://192.168.102.19:7474/
Bolt URI: bolt://192.168.102.19:7687
user: neo4j
```

Python Driver 使用 Bolt URI，不使用 Browser URL。

In [14]:
RAG_NEO4J = {
    'uri': os.getenv('RAG_NEO4J_URI', 'bolt://192.168.102.19:7687'),
    'user': os.getenv('RAG_NEO4J_USER', 'neo4j'),
    'password': os.getenv('RAG_NEO4J_PASSWORD', 'neo4j@2025'),
}

safe_config = dict(RAG_NEO4J)
safe_config['password'] = '***'
pprint(safe_config)

driver = GraphDatabase.driver(
    RAG_NEO4J['uri'],
    auth=(RAG_NEO4J['user'], RAG_NEO4J['password']),
    connection_timeout=10,
)

try:
    with driver.session() as session:
        result = session.run('RETURN 1 AS ok').single()
    NEO4J_READY = True
    print('Neo4j connected:', result['ok'])
except Exception as exc:
    NEO4J_READY = False
    print('Neo4j connection failed:', type(exc).__name__, exc)

{'password': '***', 'uri': 'bolt://192.168.102.19:7687', 'user': 'neo4j'}
Neo4j connected: 1


## 6. 创建约束

约束的作用类似关系数据库里的唯一索引。重复运行 notebook 时，约束配合 `MERGE` 可以避免重复创建同一节点。

In [15]:
CONSTRAINTS = [
    'CREATE CONSTRAINT rag_document_id IF NOT EXISTS FOR (d:Document) REQUIRE d.doc_id IS UNIQUE',
    'CREATE CONSTRAINT rag_chunk_id IF NOT EXISTS FOR (c:Chunk) REQUIRE c.chunk_id IS UNIQUE',
    'CREATE CONSTRAINT rag_entity_name IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE',
]

RUN_SCHEMA_SETUP = False

if not NEO4J_READY:
    print('skip constraints: Neo4j is not connected')
elif not RUN_SCHEMA_SETUP:
    print('skip constraints: set RUN_SCHEMA_SETUP = True to create constraints')
else:
    with driver.session() as session:
        for cypher in CONSTRAINTS:
            session.run(cypher)
            print('applied:', cypher)

skip constraints: set RUN_SCHEMA_SETUP = True to create constraints


## 7. 写入函数设计

写入时使用 `MERGE`，避免重复运行产生大量重复节点。

流程：

```text
MERGE Document
MERGE Chunk
MERGE Entity(subject)
MERGE Entity(object)
MERGE Document-HAS_CHUNK-Chunk
MERGE Chunk-MENTIONS-Entity
MERGE Entity-RELATION-Entity
```

In [16]:
def sanitize_relation_type(predicate: str) -> str:
    relation_type = ''.join(ch if ch.isalnum() else '_' for ch in predicate.strip())
    relation_type = relation_type.strip('_') or 'RELATED_TO'
    return relation_type[:40]


def build_graph_record(triple: dict, chunks_by_id: dict) -> dict:
    chunk = chunks_by_id.get(triple['chunk_id'], {})
    return {
        'doc_id': triple['doc_id'],
        'file_name': triple.get('file_name') or chunk.get('file_name', ''),
        'chunk_id': triple['chunk_id'],
        'page_start': triple.get('page_start') or chunk.get('page_start'),
        'page_end': triple.get('page_end') or chunk.get('page_end'),
        'chunk_text': chunk.get('text', ''),
        'subject': triple['subject'],
        'predicate': triple['predicate'],
        'relation_type': sanitize_relation_type(triple['predicate']),
        'object': triple['object'],
        'evidence': triple.get('evidence', ''),
        'confidence': float(triple.get('confidence', 0.5)),
    }

records = [build_graph_record(triple, chunks_by_id) for triple in triples]

print('record_count:', len(records))
pprint(records[:2])

record_count: 23
[{'chunk_id': '63b7d4d0675426b5_chunk_0024',
  'chunk_text': '③3 年一遇洪水调度示例：\n'
                '结合3 日内气象预报结果，开展洪水预报作业，根据洪\n'
                '水预报结果，提前两天开始预泄，预泄流量200m3/s，预泄\n'
                '水量0.35 亿m3。\n'
                '3 年一遇洪水控泄流量200m3/s，最高水位152.93m，最\n'
                '高水位出现在207h，调度过程见图3。\n'
                '\n'
                '图 3 3 年一遇洪水控泄200 m3/s\n'
                '\n'
                '（2）5-10 年一遇洪水调度\n'
                '结合3 日内气象预报结果，开展洪水预报，经综合会商\n'
                '\n'
                '后确定洪水预泄调度措施，为了满足水库汛限水位管理要\n'
                '\n'
                '求，按照152.00m 为起调水位，提前2 天开始预泄，5 年一\n'
                '遇洪水预泄流量100m3/s，10 年一遇洪水预泄流量200 m3/s，\n'
                '预泄水量分别为0.17 亿m3 和0.35 亿m3。\n'
                '对5 年一遇的设计洪水，下泄流量分别为100m3/s、\n'
                '200m3/s、300m3/s 进行调洪计算；对10 年一遇的设计洪水，\n'
                '下泄流量分别为300m3/s、400m3/s 进行调洪计算，结果如表\n'
                '5，调洪调度示例见表6，图4。\n'
                '\n'
                '15',
  'confidence': 0.95,
  'doc_id':

## 8. Cypher 写入语句

Neo4j 的关系类型不能用参数传入，所以这里会按每条 triple 的 `predicate` 生成安全的关系类型。

同时额外保存一个 `type` 属性，保留原始中文谓词。

In [17]:
def write_graph_record(tx, record: dict):
    relation_type = record['relation_type']
    cypher = f'''
    MERGE (d:Document {{doc_id: $doc_id}})
      SET d.file_name = $file_name
    MERGE (c:Chunk {{chunk_id: $chunk_id}})
      SET c.doc_id = $doc_id,
          c.page_start = $page_start,
          c.page_end = $page_end,
          c.text = $chunk_text,
          c.file_name = $file_name
    MERGE (d)-[:HAS_CHUNK]->(c)
    MERGE (s:Entity {{name: $subject}})
    MERGE (o:Entity {{name: $object}})
    MERGE (c)-[:MENTIONS]->(s)
    MERGE (c)-[:MENTIONS]->(o)
    MERGE (s)-[r:{relation_type} {{evidence_chunk_id: $chunk_id, object_name: $object}}]->(o)
      SET r.type = $predicate,
          r.evidence = $evidence,
          r.confidence = $confidence,
          r.doc_id = $doc_id,
          r.page_start = $page_start,
          r.page_end = $page_end
    RETURN id(r) AS relation_id
    '''
    return tx.run(cypher, record).single()['relation_id']

print('write function ready')

write function ready


## 9. 可选：写入 Neo4j

默认关闭写入：

```python
RUN_NEO4J_WRITE = False
```

确认 `records` 质量后，再改为 `True`。

In [18]:
RUN_NEO4J_WRITE = True

if not NEO4J_READY:
    print('skip write: Neo4j is not connected')
elif not RUN_NEO4J_WRITE:
    print('skip write: set RUN_NEO4J_WRITE = True to write graph')
else:
    relation_ids = []
    with driver.session() as session:
        for record in records:
            relation_id = session.execute_write(write_graph_record, record)
            relation_ids.append(relation_id)
    print('written relations:', len(relation_ids))
    print('relation_ids:', relation_ids[:10])

Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=22, column=12, offset=751>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 751, 'line': 22, 'column': 12}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MERGE (d:Document {doc_id: $doc_id})\n      SET d.file_name = $file_name\n    MERGE (c:Chunk {chunk_id: $chunk_id})\n      SET c.doc_id = $doc_id,\n          c.page_start = $page_start,\n          c.page_end = $page_end,\n          c.text = $chunk_text,\n          c.file_name = $file_name\n    MERGE (d)-[:HAS_CHUNK]->(c)\n    MERGE 

written relations: 23
relation_ids: [50673, 50676, 50679, 50679, 50679, 50679, 50679, 50679, 50679, 50679]


## 10. 查询验证

写入后，可以用下面查询查看某个实体的一跳关系。

如果没有执行写入，这里可能查不到结果。

In [19]:
def query_entity_relations(entity_name: str, limit: int = 20) -> list[dict]:
    cypher = '''
    MATCH (e:Entity {name: $entity_name})-[r]-(other:Entity)
    RETURN e.name AS entity,
           type(r) AS relation_type,
           r.type AS predicate,
           other.name AS other,
           r.evidence AS evidence,
           r.evidence_chunk_id AS chunk_id,
           r.confidence AS confidence
    LIMIT $limit
    '''
    with driver.session() as session:
        return [dict(record) for record in session.run(cypher, entity_name=entity_name, limit=limit)]

if not NEO4J_READY:
    print('skip query: Neo4j is not connected')
else:
    entity_name = records[0]['subject'] if records else '北京市密云水库'
    results = query_entity_relations(entity_name)
    print('entity:', entity_name)
    print('relation_count:', len(results))
    pprint(results[:5])

entity: 3 年一遇洪水调度示例
relation_count: 1
[{'chunk_id': '63b7d4d0675426b5_chunk_0024',
  'confidence': 0.95,
  'entity': '3 年一遇洪水调度示例',
  'evidence': '结合3 日内气象预报结果，开展洪水预报作业，根据洪',
  'other': '3 日内气象预报结果, 阐述3 日内气象预报结果',
  'predicate': '结合',
  'relation_type': '结合'}]


## 11. 图谱计数检查

写入后，可以查看当前图谱节点和关系数量。

In [20]:
def graph_counts() -> dict:
    node_cypher = '''
    MATCH (n)
    WITH labels(n)[0] AS label, count(*) AS count
    RETURN collect({label: label, count: count}) AS node_counts
    '''
    rel_cypher = 'MATCH ()-[r]->() RETURN count(r) AS relationship_count'
    with driver.session() as session:
        node_counts = session.run(node_cypher).single()['node_counts']
        relationship_count = session.run(rel_cypher).single()['relationship_count']
    return {'node_counts': node_counts, 'relationship_count': relationship_count}

if not NEO4J_READY:
    print('skip counts: Neo4j is not connected')
else:
    pprint(graph_counts())

{'node_counts': [{'count': 17671, 'label': '240'},
                 {'count': 667, 'label': '239'},
                 {'count': 16, 'label': 'lin_test'},
                 {'count': 7192, 'label': 'BaseNode'},
                 {'count': 44, 'label': '123'},
                 {'count': 8, 'label': 'hh_test'},
                 {'count': 2, 'label': 'Document'},
                 {'count': 3, 'label': 'Chunk'},
                 {'count': 17, 'label': 'Entity'}],
 'relationship_count': 50692}


## 12. 本课小结

本课完成：

```text
triples.json
-> graph records
-> Neo4j nodes and relationships
-> one-hop relation query
```

关键结论：

```text
图谱关系必须保留 evidence 和 chunk_id。
没有证据的图谱关系不适合用于问答。
```

下一课会进入：

```text
BM25 召回 + 向量召回 + Neo4j 图检索
-> evidence package
```

## 13. 练习

请你回答：

1. 为什么写入 Neo4j 要用 `MERGE`，而不是只用 `CREATE`？
2. 为什么 Entity 节点只用 `name` 做第一版唯一键会有局限？
3. 为什么关系上要保存 `evidence_chunk_id`？
4. ES 的 `chunk_id` 和 Neo4j 的 `Chunk.chunk_id` 为什么必须一致？